###Read xlsx bdx files from volumes

In [0]:
# ENABLE_HEADER_SHIFTING = True   # Toggle header row alignment logic
# ENABLE_COLUMN_PADDING = True   # Toggle padding for uneven row lengths
# ENABLE_GPT_CACHING = True      # Toggle GPT caching layer
# ENABLE_GPT_ENRICHMENT = True   # Toggle enriched descriptions with glossary


In [0]:
def get_langchain_gpt_schema_chain(client):
    from langchain.chains.openai_functions import create_structured_output_chain
    from langchain.prompts import ChatPromptTemplate
    from langchain.chat_models import ChatOpenAI

    prompt = ChatPromptTemplate.from_template("""
You are an expert at interpreting Excel data layouts.

You will be given a tab-separated version of the top of a spreadsheet. Your job is to identify the structure:
- Clean column headers
- Determine where real tabular data starts
- Output a schema summary including semantic data types

{raw_text}
    """)

    from langchain_core.output_parsers import JsonOutputKeyTools
    from langchain_core.utils.function_calling import convert_to_openai_function
    output_format = convert_to_openai_function(data_dict_function)
    chain = create_structured_output_chain(output_format, prompt, llm=client)
    return chain

In [0]:
%sql
USE CATALOG bdx;
use schema data_dictionary;

In [0]:
%pip install openai openpyxl tiktoken
%pip install -U mlflow
dbutils.library.restartPython()

  Obtaining dependency information for openai from https://files.pythonhosted.org/packages/80/9a/f34f163294345f123673ed03e77c33dee2534f3ac1f9d18120384457304d/openai-1.75.0-py3-none-any.whl.metadata
  Obtaining dependency information for openpyxl from https://files.pythonhosted.org/packages/c0/da/977ded879c29cbd04de313843e76868e6e13408a94ed6b987245dc7c8506/openpyxl-3.1.5-py2.py3-none-any.whl.metadata
  Obtaining dependency information for tiktoken from https://files.pythonhosted.org/packages/b1/73/41591c525680cd460a6becf56c9b17468d3711b1df242c53d2c7b2183d16/tiktoken-0.9.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for anyio<5,>=3.5.0 from https://files.pythonhosted.org/packages/a1/ee/48ca1a7c89ffec8b6a0c5d02b89c305671d5ffd8d3c94acf8b8c408575bb/anyio-4.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for httpx<1,>=0.23.0 from https://files.pythonhosted.org/packages/2a/39/e50c7c3a983047577ee07d2a9e53faf5a69493943e

In [0]:
import mlflow
mlflow.openai.autolog()

In [0]:
from openai import OpenAI
from openai import AzureOpenAI
import os
import os
import json
import pandas as pd
from openai import OpenAI

endpoint = "https://hanna-m9ic9273-eastus2.cognitiveservices.azure.com/"
model_name = "gpt-4.1"
deployment = "gpt-4.1"

subscription_key = "C9oe9lxteRbZXvRHdkXRq7uezqsl1bDWQ7tJH91uAHaWbjnpoQ8rJQQJ99BDACHYHv6XJ3w3AAAAACOGb14W"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

In [0]:
def clean_percent(val):
    """
    Cleans percentage values like '45.8%', 'N/A', or None to float.
    """
    try:
        if isinstance(val, str):
            val = val.replace("%", "").strip()
        return float(val)
    except:
        return None


In [0]:
def flatten_xlsx_from_volume(volume_folder):
    """
    Ingest Excel files from Unity Catalog and flatten each sheet into a tab-separated format using GPT for:
    - Header reconstruction
    - Data row detection
    - Fallback flattening if GPT fails
    """
    from collections import defaultdict
    import pandas as pd

    all_text_blocks = {}
    flatten_stats = defaultdict(list)

    try:
        full_paths = [
            f.path for f in dbutils.fs.ls(volume_folder)
            if f.path.lower().endswith(".xlsx")
        ]
    except Exception as e:
        print(f"❌ Failed to list files in {volume_folder} — {e}")
        return all_text_blocks

    if not full_paths:
        print(f"ℹ️ No Excel files found in {volume_folder}")
        return all_text_blocks

    print(f"📊 Found {len(full_paths)} Excel files to process")

    for dbfs_path in full_paths:
        file_name = Path(dbfs_path).name
        print(f"📥 Processing: {file_name}")

        try:
            local_path = copy_volume_file_to_tmp_via_spark(dbfs_path)
            with pd.ExcelFile(local_path) as xls:
                for sheet in xls.sheet_names:
                    key = f"{file_name}::{sheet}"
                    try:
                        df = pd.read_excel(xls, sheet_name=sheet, header=None)
                        if df.empty or df.shape[1] < 2:
                            print(f"⚠️ Skipping empty or malformed sheet: {sheet}")
                            flatten_stats["skipped"].append(key)
                            continue

                        # Convert to raw text
                        sheet_text = "\n".join([
                            "\t".join("" if pd.isna(cell) else str(cell).strip() for cell in row)
                            for row in df.values if any(pd.notna(row))
                        ])

                        # 💡 Use GPT to determine headers + start
                        result = generate_rich_data_dictionary(sheet_text, client)
                        headers = result.get("column_headers", [])
                        data_start = result.get("data_start_row", 0)

                        # Slice data using GPT's suggestion
                        data_rows = df.iloc[data_start:].reset_index(drop=True)

                        if data_rows.empty or len(headers) != data_rows.shape[1]:
                            raise ValueError("Header length mismatch or empty result")

                        data_rows.columns = headers

                        text = "\n".join([
                            "\t".join("" if pd.isna(cell) else str(cell).strip() for cell in row)
                            for row in data_rows.values
                        ])
                        all_text_blocks[key] = text
                        flatten_stats["structured"].append(key)
                        print(f"✅ GPT-guided flatten: {key} ({len(data_rows)} rows)")

                    except Exception as structured_e:
                        # === Fallback to raw flattening ===
                        try:
                            print(f"  🔁 Fallback flatten due to: {structured_e}")
                            df = pd.read_excel(xls, sheet_name=sheet, header=None)
                            raw_text = "\n".join([
                                "\t".join("" if pd.isna(cell) else str(cell).strip() for cell in row)
                                for row in df.values if any(pd.notna(row))
                            ])
                            all_text_blocks[key] = raw_text
                            flatten_stats["fallback"].append(key)
                            print(f"  ✅ Raw fallback flatten: {key} ({len(df)} rows)")
                        except Exception as fallback_error:
                            flatten_stats["failed"].append(key)
                            print(f"  ❌ Fallback failed for {sheet}: {fallback_error}")

        except Exception as file_error:
            print(f"❌ File-level error for {file_name}: {file_error}")

    print(f"\n📊 Flattening Summary:")
    print(f"  ✅ Structured: {len(flatten_stats['structured'])}")
    print(f"  🔁 Fallback  : {len(flatten_stats['fallback'])}")
    print(f"  ⚠️ Skipped   : {len(flatten_stats['skipped'])}")
    print(f"  ❌ Failed    : {len(flatten_stats['failed'])}")

    return all_text_blocks

In [0]:
# def extract_headers_and_data_start(raw_text, client):
#     """
#     Uses GPT to infer structured headers and the starting row for actual data
#     from messy Excel text.
    
#     Args:
#         raw_text (str): Flattened text from Excel sheet
#         client: OpenAI client
    
#     Returns:
#         dict: {
#             'column_headers': [...],
#             'data_start_row': int
#         }
#     """
#     prompt = f"""
# You are given messy tab-separated text from an Excel sheet. It may contain multiple rows of headers,
# notes, or descriptive text before the actual data begins.

# Your job is to:
# 1. Identify the clean, final list of column headers.
# 2. Indicate which row (0-indexed) the actual data starts on.

# Text (first 8000 chars only):
# {raw_text[:8000]}

# Return JSON with keys:
# - column_headers: list of strings
# - data_start_row: integer (first row of actual data)
# """

#     response = client.chat.completions.create(
#         model="gpt-4.1",
#         messages=[{"role": "user", "content": prompt}]
#     )

#     # Attempt to parse clean JSON response
#     try:
#         return json.loads(response.choices[0].message.content)
#     except Exception as e:
#         print(f"⚠️ Failed to parse GPT response: {e}")
#         return {
#             "column_headers": [],
#             "data_start_row": 0
#         }


In [0]:
def gpt_guided_dataframe_from_xlsx(file_path: str, sheet_name: str, raw_text: str, client, debug=False) -> pd.DataFrame:
    import pandas as pd
    from collections import Counter
    import numpy as np
    import re

    df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    raw_rows = df_raw.values.tolist()

    # === Step 1: GPT schema inference
    schema = generate_rich_data_dictionary(raw_text, client)
    gpt_headers = schema.get("column_headers", [])
    data_start_row = schema.get("data_start_row", 0)
    col_count = len(gpt_headers)

    if debug:
        print(f"🧠 GPT headers: {gpt_headers}")
        print(f"🟢 Data starts at row: {data_start_row}")

    # === Step 2: Slice rows GPT believes are data
    data_rows = raw_rows[data_start_row:]
    if not data_rows:
        raise ValueError("No data rows found after GPT start row.")

    # === Step 3: Smart alignment using blank offset
    def leading_blanks(row):
        return next((i for i, cell in enumerate(row) if str(cell).strip()), len(row))

    offset_counts = Counter([leading_blanks(row) for row in data_rows if any(pd.notna(row))])
    common_offset = offset_counts.most_common(1)[0][0]

    aligned = []
    for row in data_rows:
        trimmed = row[common_offset:]
        trimmed = trimmed[:col_count] + [""] * (col_count - len(trimmed))
        aligned.append([str(cell).strip() if cell is not None else "" for cell in trimmed])

    df_aligned = pd.DataFrame(aligned, columns=gpt_headers)

    if df_aligned.shape[1] != col_count:
        raise ValueError("Column count mismatch after alignment.")

    return df_aligned


In [0]:
# 📦 Caching GPT responses to avoid duplicate costs
gpt_schema_cache = {}


In [0]:
data_dict_function = {
    "name": "generate_rich_data_dictionary",
    "description": "Analyze raw Excel/CSV sheet and return standardized column metadata and structural layout.",
    "parameters": {
        "type": "object",
        "properties": {
            "columns": {
                "type": "array",
                "description": "List of detected columns with metadata",
                "items": {
                    "type": "object",
                    "properties": {
                        "original_column_name": {"type": "string"},
                        "standardized_column_name": {"type": "string"},
                        "description": {"type": "string"},
                        "inferred_data_type": {"type": "string"},
                        "percent_null": {"type": "number"},
                        "examples": {"type": "array", "items": {"type": "string"}},
                        "semantic_score": {"type": "number"}
                    },
                    "required": [
                        "original_column_name",
                        "standardized_column_name",
                        "description",
                        "inferred_data_type",
                        "percent_null",
                        "examples",
                        "semantic_score"
                    ]
                }
            },
            "column_headers": {
                "type": "array",
                "description": "Final cleaned and standardized column headers in order",
                "items": {"type": "string"}
            },
            "data_start_row": {
                "type": "integer",
                "description": "Row number (0-indexed) where actual data starts"
            }
        },
        "required": ["columns", "column_headers", "data_start_row"]
    }
}



In [0]:
def generate_rich_data_dictionary(raw_text, client, file_name=None, sheet_name=None):
    from hashlib import sha256
    import json

    # === Step 1: Token-efficient sampling ===
    lines = raw_text.strip().split("\n")
    header_lines = lines[:6]
    sample_data_lines = lines[6:16]
    prompt_text = "\n".join(header_lines + sample_data_lines)

    # === Step 2: Caching ===
    header_hash = sha256(prompt_text.encode()).hexdigest()
    if header_hash in gpt_schema_cache:
        return gpt_schema_cache[header_hash]

    # === Step 3: Add metadata context ===
    context = ""
    if file_name or sheet_name:
        context += f"This file is named '{file_name}' and sheet is '{sheet_name}'. It may contain financial, tabular, or projection-related data.\n"

    # === Step 4: Optimized GPT Prompt ===
    prompt = f"""
You are an expert financial data analyst skilled at cleaning spreadsheet data.

{context}
You are analyzing a messy Excel or CSV sheet that was flattened into tab-separated text.

This sheet contains:
- Notes or metadata above the table
- Multiple stacked header rows (2 to 4 rows)
- Merged cells that split headers over several rows

Your job is to:
1. Identify the first row that contains real tabular data (ignore top notes)
2. Merge all relevant header rows into a single list of clear, standardized column headers
3. Clean and normalize header text to be consistent and meaningful

📌 Additionally:
- Use **standardized financial terminology** for column headers when possible.
- For example:
    - "CY2023 Est" → "2023_Estimate"
    - "Q1 2024" → "2024_Q1"
    - "2024 Av Commission" → "2024_Average_Commission"
    - "2024 Treaty Year Estimate" → "2024_Treaty_Year_Estimate"
    - "Class" → "Business_Line" or "Segment" if applicable

Return:
- "columns": a list of column metadata objects with:
    - original_column_name
    - standardized_column_name
    - description
    - inferred_data_type
    - percent_null
    - examples
    - semantic_score
- "column_headers": the cleaned, standardized header names in order
- "data_start_row": the 0-based row index where actual tabular data starts

Be smart and careful:
- Distinguish between notes vs headers vs data rows
- Resolve multi-row headers with merged cells intelligently
- Skip total/footer rows that are not part of the core data table
Text:
{prompt_text}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=[{"role": "user", "content": prompt}],
            functions=[data_dict_function],
            function_call={"name": "generate_rich_data_dictionary"},
            temperature=0.0  # ⬅️ deterministic output
        )

        result = json.loads(response.choices[0].message.function_call.arguments)
        gpt_schema_cache[header_hash] = result  # 🧠 Cache result
        return result

    except Exception as e:
        print(f"❌ GPT schema extraction failed: {e}")
        return {
            "columns": [],
            "column_headers": [],
            "data_start_row": 0
        }


In [0]:
import os
from pathlib import Path

def copy_volume_file_to_tmp_via_spark(volume_path: str) -> str:
    """
    Copies a file from Unity Catalog volume to /tmp/ using Spark's binaryFile format.

    Args:
        volume_path (str): Full DBFS/Volume path to the file

    Returns:
        str: Local /tmp path where the file is saved

    Raises:
        FileNotFoundError: If the file doesn't exist at the given volume path
        IOError: If there's an error reading or writing the file
    """
    file_name = Path(volume_path).name
    tmp_path = f"/tmp/{file_name}"

    # Validate file existence
    try:
        dbutils.fs.ls(volume_path)
    except Exception:
        raise FileNotFoundError(f"🚫 File not found: {volume_path}")

    try:
        binary_df = spark.read.format("binaryFile").load(volume_path)
        content_rows = binary_df.select("content").collect()

        if not content_rows or not content_rows[0]["content"]:
            raise IOError(f"⚠️ File is empty or unreadable: {volume_path}")

        with open(tmp_path, "wb") as f:
            f.write(content_rows[0]["content"])

        print(f"✅ File copied to: {tmp_path}")
        return tmp_path

    except Exception as e:
        raise IOError(f"❌ Failed to copy from volume: {volume_path} — {str(e)}")


In [0]:
import pandas as pd
import os

def flatten_xlsx_from_volume(volume_folder):
    """
    For each .xlsx file in a Unity Catalog Volume folder:
    - Copies to /tmp
    - Reads with pandas
    - Returns dict of {filename::sheet_name: raw structured text (tab-separated)}
    """
    all_text_blocks = {}
    full_paths = [
        f.path for f in dbutils.fs.ls(volume_folder)
        if f.name.endswith(".xlsx")
    ]

    for dbfs_path in full_paths:
        print(f"📥 Processing: {dbfs_path}")
        try:
            local_path = copy_volume_file_to_tmp_via_spark(dbfs_path)
            xls = pd.ExcelFile(local_path)

            for sheet in xls.sheet_names:
                df = pd.read_excel(xls, sheet_name=sheet, header=None)
                sheet_text = "\n".join(["\t".join(map(str, row)) for row in df.values])
                key = f"{os.path.basename(dbfs_path)}::{sheet}"
                all_text_blocks[key] = sheet_text
                print(f"  ✅ Flattened: {key}")

        except Exception as e:
            print(f"  ❌ Failed: {dbfs_path} — {e}")

    return all_text_blocks

In [0]:
from pyspark.sql.utils import AnalysisException
from pyspark.sql import DataFrame

# Fully qualify if working in Unity Catalog
# schema_table_name = "bdx.data_dictionary.gpt_data_dictionary"  # or 'catalog.schema.gpt_data_dictionary'

# # Flag for tracking table state
# table_created = False


In [0]:
# from datetime import datetime

# def process_data_dictionaries(raw_text_map, schema_table_name):
#     """
#     Processes raw text data into structured data dictionaries and writes to a Delta table.
    
#     Args:
#         raw_text_map (dict): Dictionary of {sheet_key: raw_text}
#         schema_table_name (str): Full name of the Delta table (e.g., 'bdx.cleaned_schema')

#     Returns:
#         dict: Map of processed data dictionaries, keyed by sheet_key
#     """
#     processed_data_dictionary_map = {}
#     total_sheets = len(raw_text_map)
#     processed_count = 0
    
#     print(f"🔍 Processing {total_sheets} sheets to extract data dictionaries")

#     # Check if Delta table already exists
#     table_exists = False
#     try:
#         spark.sql("DESCRIBE TABLE bdx.data_dictionary.gpt_data_dictionary")
#         table_exists = True
#         print(f"📊 Table {schema_table_name} exists — will update entries")
#     except:
#         print(f"📊 Table {schema_table_name} will be created")

#     # Buffer for batch insertion
#     all_flattened_records = []

#     for sheet_key, raw_text in raw_text_map.items():
#         print(f"\n🧠 Extracting schema for: {sheet_key}")
#         try:
#             # Step 1: Extract metadata using LLM
#             data_dictionary = generate_rich_data_dictionary(raw_text,client)
#             processed_data_dictionary_map[sheet_key] = data_dictionary

#             # Step 2: Flatten structure
#             sheet_records = [
#             {
#                 "file_name": sheet_key.split("::")[0],
#                 "sheet_name": sheet_key.split("::")[1],
#                 "sheet_key": sheet_key,
#                 "column_name": col.get("column_name"),
#                 "description": col.get("description"),
#                 "inferred_data_type": col.get("inferred_data_type"),
#                 "percent_null": clean_percent(col.get("percent_null")),  # ✅ fixed
#                 "examples": ", ".join(str(x) for x in col.get("examples", [])),
#                 "semantic_score": col.get("semantic_score"),
#                 "processed_at": str(datetime.utcnow().isoformat())
#             }
#             for col in data_dictionary.get("columns", [])
#         ]


#             all_flattened_records.extend(sheet_records)
#             processed_count += 1
#             print(f"✅ Processed {sheet_key}: {len(sheet_records)} columns")

#         except Exception as e:
#             print(f"❌ Failed on {sheet_key} — {str(e)}")

#     # Step 3: Save to Delta if records found
#     from pyspark.sql.types import DoubleType
#     if all_flattened_records:
#         try:
#             # Convert to Spark DataFrame
#             new_data_df = spark.createDataFrame(pd.DataFrame(all_flattened_records))

#             # 🔧 Fix: Enforce type consistency for 'percent_null'
#             if "percent_null" in new_data_df.columns:
#                 new_data_df = new_data_df.withColumn("percent_null", new_data_df["percent_null"].cast(DoubleType()))

#             if table_exists:
#                 existing_df = spark.table(schema_table_name)
#                 sheet_keys = [r["sheet_key"] for r in all_flattened_records]
#                 filtered_existing = existing_df.filter(~existing_df.sheet_key.isin(sheet_keys))
#                 final_df = filtered_existing.unionByName(new_data_df)
#                 final_df.write.format("delta").mode("overwrite").saveAsTable(schema_table_name)
#             else:
#                 new_data_df.write.format("delta").mode("overwrite").saveAsTable(schema_table_name)

#             print(f"\n✅ Saved {processed_count}/{total_sheets} sheets to: {schema_table_name}")

#         except Exception as e:
#             print(f"\n❌ Failed writing to Delta table: {e}")
#             raise
#     else:
#         print("\n⚠️ No valid records to write")

#     return processed_data_dictionary_map


In [0]:
from datetime import datetime
from pyspark.sql.types import DoubleType

def clean_percent(val):
    """
    Cleans percentage values like '45.8%', 'N/A', or None to float.
    """
    try:
        if isinstance(val, str):
            val = val.replace("%", "").strip()
        return float(val)
    except:
        return None
schema_table_name = "bdx.data_dictionary.gpt_data_dictionary"  #table name in Unity Catalog
def process_data_dictionaries(raw_text_map, schema_table_name):
    """
    Processes raw text data into structured data dictionaries and writes to a Delta table.

    Args:
        raw_text_map (dict): Dictionary of {sheet_key: raw_text}
        schema_table_name (str): Full name of the Delta table (e.g., 'bdx.cleaned_schema')

    Returns:
        dict: Map of processed data dictionaries, keyed by sheet_key
    """
    processed_data_dictionary_map = {}
    total_sheets = len(raw_text_map)
    processed_count = 0

    print(f"🔍 Processing {total_sheets} sheets to extract data dictionaries")

    # Check if Delta table already exists
    table_exists = False
    try:
        spark.sql(f"DESCRIBE TABLE {schema_table_name}")
        table_exists = True
        print(f"📊 Table {schema_table_name} exists — will update entries")
    except:
        print(f"📊 Table {schema_table_name} will be created")

    # Buffer for batch insertion
    all_flattened_records = []

    for sheet_key, raw_text in raw_text_map.items():
        print(f"\n🧠 Extracting schema for: {sheet_key}")
        try:
            # Step 1: Extract metadata using LLM
            data_dictionary = generate_rich_data_dictionary(raw_text, client)
            processed_data_dictionary_map[sheet_key] = data_dictionary

            # Step 2: Flatten structure
            sheet_records = [
            {
                "file_name": sheet_key.split("::")[0],
                "sheet_name": sheet_key.split("::")[1],
                "sheet_key": sheet_key,
                "original_column_name": col.get("original_column_name", col.get("column_name", "")),
                "standardized_column_name": col.get("standardized_column_name", col.get("column_name", "")),
                "description": col.get("description", ""),
                "description_enriched": col.get("description_enriched", ""),
                "inferred_data_type": col.get("inferred_data_type", "unknown"),
                "percent_null": clean_percent(col.get("percent_null", 0)),
                "examples": ", ".join(str(x) for x in col.get("examples", [])),
                "semantic_score": col.get("semantic_score", None),
                "processed_at": str(datetime.utcnow().isoformat())
            }

                for col in data_dictionary.get("columns", [])
            ]

            all_flattened_records.extend(sheet_records)
            processed_count += 1
            print(f"✅ Processed {sheet_key}: {len(sheet_records)} columns")

        except Exception as e:
            print(f"❌ Failed on {sheet_key} — {str(e)}")

    # Step 3: Save to Delta if records found
    if all_flattened_records:
        try:
            new_data_df = spark.createDataFrame(pd.DataFrame(all_flattened_records))

            # 🔧 Enforce type consistency for 'percent_null'
            if "percent_null" in new_data_df.columns:
                new_data_df = new_data_df.withColumn("percent_null", new_data_df["percent_null"].cast(DoubleType()))

            if table_exists:
                existing_df = spark.table(schema_table_name)
                sheet_keys = [r["sheet_key"] for r in all_flattened_records]
                filtered_existing = existing_df.filter(~existing_df.sheet_key.isin(sheet_keys))
                final_df = filtered_existing.unionByName(new_data_df)
                final_df.write.format("delta").mode("overwrite").saveAsTable(schema_table_name)
            else:
                new_data_df.write.format("delta").mode("overwrite").saveAsTable(schema_table_name)

            print(f"\n✅ Saved {processed_count}/{total_sheets} sheets to: {schema_table_name}")

        except Exception as e:
            print(f"\n❌ Failed writing to Delta table: {e}")
            print("🧪 Schema of new_data_df:")
            new_data_df.printSchema()
            raise
    else:
        print("\n⚠️ No valid records to write")

    return processed_data_dictionary_map


In [0]:
volume_folder = "dbfs:/Volumes/test/bronze/raw/"
raw_text_map = flatten_xlsx_from_volume(volume_folder)

📥 Processing: dbfs:/Volumes/test/bronze/raw/test2.xlsx
✅ File copied to: /tmp/test2.xlsx
  ✅ Flattened: test2.xlsx::Sheet1


In [0]:
schema_table_name = "gpt_data_dictionary"
processed_data_dictionary_map = process_data_dictionaries(raw_text_map, schema_table_name)

🔍 Processing 1 sheets to extract data dictionaries
📊 Table gpt_data_dictionary exists — will update entries

🧠 Extracting schema for: test2.xlsx::Sheet1
✅ Processed test2.xlsx::Sheet1: 13 columns

✅ Saved 1/1 sheets to: gpt_data_dictionary


Trace(request_id=tr-2ab0da244f2b4ff59e12d6b1332c9ce7)

In [0]:
if "processed_data_dictionary_map" not in globals():
    processed_data_dictionary_map = {}

failed_sheets = []
permanently_failed_sheets = []

def run_schema_extraction(sheet_key, raw_text, client, force_mode=False):
    try:
        if force_mode:
            # Optionally adjust prompt if retrying
            raw_text = f"WARNING: Previous GPT parse failed. Force structured schema.\n\n{raw_text}"

        result = generate_rich_data_dictionary(raw_text, client)

        processed_data_dictionary_map[sheet_key] = result
        print(f"✅ Extracted schema for: {sheet_key}")
        return True

    except Exception as e:
        print(f"❌ Extraction failed for {sheet_key} — {str(e)}")
        return False

# Retry-enabled loop
for sheet_key, raw_text in raw_text_map.items():
    if sheet_key in processed_data_dictionary_map:
        continue

    print(f"\n🧠 Extracting schema for: {sheet_key}")
    success = run_schema_extraction(sheet_key, raw_text, client)

    if not success:
        print(f"🔁 Retrying with enhanced prompt for: {sheet_key}")
        retry_success = run_schema_extraction(sheet_key, raw_text, client, force_mode=True)

        if not retry_success:
            permanently_failed_sheets.append(sheet_key)


In [0]:
from typing import List

def enrich_column_descriptions(columns: List[dict], client) -> List[dict]:
    """
    Use GPT-4.1 to rewrite/improve column descriptions.
    """
    enriched = []
    for col in columns:
        prompt = f"Improve this data column description using binders glossary:\n\n'{col['description']}'"

        try:
            response = client.chat.completions.create(
                model="gpt-4.1",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            new_desc = response.choices[0].message.content.strip()
            col["description_enriched"] = new_desc
        except Exception as e:
            col["description_enriched"] = col["description"]
            print(f"❌ GPT failed for column '{col['column_name']}': {e}")

        enriched.append(col)
    return enriched

In [0]:
# Apply GPT enrichment to all processed columns
for sheet_key, schema in processed_data_dictionary_map.items():
    columns = schema.get("columns", [])
    enriched = enrich_column_descriptions(columns, client)
    processed_data_dictionary_map[sheet_key]["columns"] = enriched

print(f"✅ Column descriptions enriched for {len(processed_data_dictionary_map)} sheets.")

✅ Column descriptions enriched for 1 sheets.


[Trace(request_id=tr-fe414ed36ee7463c8b23e1a2e595f156), Trace(request_id=tr-6d5c4aaa258341b2b5fd5e522f28ba6f), Trace(request_id=tr-d1d9ad60ceb940ddaba26f84b2e422d9), Trace(request_id=tr-357e226edd2942cd9d6f149f76723cb1), Trace(request_id=tr-8a9543d0af0e437dafc434cf42e889d2), Trace(request_id=tr-c293fe985d4443ffa049743d1ed982bb), Trace(request_id=tr-962de397c7db43b69f30d5113fae3972), Trace(request_id=tr-a25269b2ac294aefb35de4c7a6726b9f), Trace(request_id=tr-c6745228d6484fd4a1c334bbce8cd9e0), Trace(request_id=tr-58a9812b43354e8392210adc2e7a9178)]

In [0]:
def gpt_smart_dataframe_from_xlsx(file_path: str, sheet_name: str, raw_text: str, client, debug=False) -> pd.DataFrame:
    import pandas as pd
    from collections import Counter
    import numpy as np
    import re
    import json
    from hashlib import sha256

    # === Step 0: Hash + GPT Caching (Performance) ===
    header_lines = raw_text.strip().split("\n")[:6]
    sample_data = raw_text.strip().split("\n")[6:16]
    prompt_snippet = "\n".join(header_lines + sample_data)
    header_hash = sha256(prompt_snippet.encode()).hexdigest()

    if header_hash in gpt_schema_cache:
        schema = gpt_schema_cache[header_hash]
        if debug:
            print("♻️ Using cached GPT schema")
    else:
        schema = generate_rich_data_dictionary(raw_text, client)
        gpt_schema_cache[header_hash] = schema  # ✅ Store

    gpt_headers = schema.get("column_headers", [])
    data_start_row = schema.get("data_start_row", 0)
    col_count = len(gpt_headers)

    if debug:
        print(f"🧠 GPT headers: {col_count} | Data starts at: {data_start_row}")

    # === Step 1: Load raw sheet ===
    try:
        df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    except Exception as e:
        raise RuntimeError(f"❌ Failed to read Excel: {e}")

    raw_rows = df_raw.values.tolist()

    # === Step 2: Extract rows after GPT-identified data start ===
    data_rows = raw_rows[data_start_row:]
    if not data_rows or all(len(r) == 0 or not any(pd.notna(cell) for cell in r) for r in data_rows):
        raise ValueError("❌ No usable data rows found")

    # === Step 3: Detect leading blanks (smart offset detection) ===
    def leading_blanks(row):
        return next((i for i, cell in enumerate(row) if str(cell).strip()), len(row))

    offset_counts = Counter([leading_blanks(r) for r in data_rows if any(pd.notna(r))])
    common_offset = offset_counts.most_common(1)[0][0]

    if debug:
        print(f"🔍 Most common leading blank columns: {common_offset}")

    # === Step 4: Align and normalize rows ===
    aligned = []
    for row in data_rows:
        trimmed = row[common_offset:]
        trimmed = trimmed[:col_count] + [""] * max(0, col_count - len(trimmed))
        aligned.append([str(cell).strip() if cell is not None else "" for cell in trimmed])

    # === Step 5: Validate header alignment ===
    df_aligned = pd.DataFrame(aligned, columns=gpt_headers)

    if df_aligned.shape[1] != col_count:
        if debug:
            print(f"⚠️ Column mismatch: GPT: {col_count}, DF: {df_aligned.shape[1]}")
        raise ValueError("⚠️ Column count mismatch — fallback triggered")

    # === Step 6: Heuristic quality check (token density) ===
    def row_token_score(row):
        return sum(1 for x in row if re.search(r"\w", str(x))) / len(row)

    scores = [row_token_score(row) for row in aligned if len(row) > 0]
    if np.mean(scores) < 0.4:
        if debug:
            print(f"⚠️ Low token density — mean: {np.mean(scores):.2f}")
        raise ValueError("⚠️ Data structure confidence too low")

    if debug:
        print(f"✅ Success: DataFrame shape = {df_aligned.shape}")

    return df_aligned




In [0]:
processed_gpt_flattened_data = {}

for sheet_key, raw_text in raw_text_map.items():
    print(f"\n🧠 Creating DataFrame using GPT-trusted headers for: {sheet_key}")
    try:
        file_name, sheet_name = sheet_key.split("::")

        # 🔄 Use Spark to copy file from volume to /tmp/
        volume_path = f"/Volumes/test/bronze/raw/{file_name}"
        file_path = copy_volume_file_to_tmp_via_spark(volume_path)

        # ✅ Use GPT-guided schema to construct final DataFrame
        df = gpt_guided_dataframe_from_xlsx(
            file_path=file_path,
            sheet_name=sheet_name,
            raw_text=raw_text,
            client=client
        )

        processed_gpt_flattened_data[sheet_key] = df
        print(f"✅ Final DataFrame for {sheet_key}: {df.shape}")

    except Exception as e:
        print(f"❌ Failed processing: {sheet_key} — {e}")



🧠 Creating DataFrame using GPT-trusted headers for: test2.xlsx::Sheet1
✅ File copied to: /tmp/test2.xlsx
✅ Final DataFrame for test2.xlsx::Sheet1: (6, 13)


In [0]:

from collections import defaultdict

def extract_values_from_dataframe(dataframe, columns):
    """
    Extract column values from a structured pandas DataFrame.

    Args:
        dataframe (pd.DataFrame): Cleaned and reconstructed sheet
        columns (List[dict]): GPT-inferred column metadata

    Returns:
        Dict[str, List[str]]: Extracted values by column
    """
    extracted = {}
    for col in columns:
        col_name = col.get("column_name")
        if not col_name or col_name not in dataframe.columns:
            extracted[col_name] = []
            continue

        col_values = dataframe[col_name].astype(str).fillna("").tolist()
        extracted[col_name] = col_values

    return extracted


In [0]:
from collections import Counter
import pandas as pd

# 🚀 NEW: Toggle for semantic filtering
USE_SEMANTIC_FILTERING = True
semantic_score_threshold = 0.2

extracted_data_map = {}
failed_sheets = []

for sheet_key, raw_text in raw_text_map.items():
    print(f"\n📄 Extracting structured data from: {sheet_key}")

    try:
        # === Step 1: Extract GPT schema ===
        schema_result = generate_rich_data_dictionary(raw_text, client)
        gpt_headers = schema_result.get("column_headers", [])
        data_start_row = schema_result.get("data_start_row", 0)
        semantic_columns = schema_result.get("columns", [])

        # ✅ Normalize header keys for safety
        for col in semantic_columns:
            col["column_name"] = col.get("standardized_column_name") or col.get("original_column_name")

        # === Step 2: Filter high-confidence columns (optional) ===
        if USE_SEMANTIC_FILTERING:
            filtered_columns = [
                col for col in semantic_columns
                if col.get("semantic_score", 0) >= semantic_score_threshold
            ]

            if len(filtered_columns) < 3:
                print(f"⚠️ Only {len(filtered_columns)} high-confidence columns for {sheet_key}. Falling back to all GPT columns.")
                filtered_columns = semantic_columns
        else:
            filtered_columns = semantic_columns

        if not filtered_columns:
            raise ValueError("❌ No usable columns found in GPT schema.")

        # === Step 3: Parse raw tab-separated data ===
        lines = raw_text.strip().split("\n")
        data_lines = lines[data_start_row:]
        rows = [line.split("\t") for line in data_lines if line.strip()]

        if not rows:
            raise ValueError("❌ No usable data rows.")

        # === Step 4: Determine dominant column count ===
        row_lengths = [len(row) for row in rows if len(row) > 1]
        most_common_length = Counter(row_lengths).most_common(1)[0][0]

        # === Step 5: Smart detection of left-shift ===
        leading_empty_counts = [
            next((i for i, cell in enumerate(row) if cell.strip()), len(row))
            for row in rows
        ]
        most_common_leading_empty = Counter(leading_empty_counts).most_common(1)[0][0]

        # === Step 6: Header realignment ===
        shift = most_common_leading_empty if most_common_leading_empty < len(gpt_headers) else 0
        gpt_headers = gpt_headers[shift:]

        if len(gpt_headers) > most_common_length:
            gpt_headers = gpt_headers[:most_common_length]
        elif len(gpt_headers) < most_common_length:
            gpt_headers += [f"col_{i}" for i in range(len(gpt_headers), most_common_length)]

        # === Step 7: Normalize and align rows ===
        cleaned_rows = []
        for row in rows:
            row_shift = shift if shift < len(row) else 0
            trimmed = row[row_shift:]
            trimmed = trimmed[:most_common_length] + [""] * (most_common_length - len(trimmed))
            cleaned_rows.append(trimmed)

        # === Step 8: Build DataFrame ===
        df = pd.DataFrame(cleaned_rows, columns=gpt_headers)
        print(f"🧠 GPT Headers: {gpt_headers}")
        print(f"✅ Filtered Columns: {[col['column_name'] for col in filtered_columns]}")
        print(f"✅ Actual DF Columns: {df.columns.tolist()}")

        # === Step 9: Extract structured columnar values ===
        values_by_column = extract_values_from_dataframe(df, filtered_columns)
        extracted_data_map[sheet_key] = values_by_column

        print(f"✅ Extracted {len(values_by_column)} columns for: {sheet_key}")

    except Exception as e:
        print(f"❌ Failed extracting {sheet_key} — {e}")
        failed_sheets.append(sheet_key)



📄 Extracting structured data from: test2.xlsx::Sheet1
🧠 GPT Headers: ['Business_Line', '2019', '2020', '2021', '2022', '2023_Estimate', '2023_Revised', '2024_Calendar_Year_Estimate', '2024_Treaty_Year_Estimate', '2024_Q1_Estimate', '2025_Q1_Estimate', '2025_Q2_Estimate', '2024_Average_Commission', 'col_13', 'col_14']
✅ Filtered Columns: ['Business_Line', '2019', '2020', '2021', '2022', '2023_Estimate', '2023_Revised', '2024_Calendar_Year_Estimate', '2024_Treaty_Year_Estimate', '2024_Q1_Estimate', '2025_Q1_Estimate', '2025_Q2_Estimate', '2024_Average_Commission']
✅ Actual DF Columns: ['Business_Line', '2019', '2020', '2021', '2022', '2023_Estimate', '2023_Revised', '2024_Calendar_Year_Estimate', '2024_Treaty_Year_Estimate', '2024_Q1_Estimate', '2025_Q1_Estimate', '2025_Q2_Estimate', '2024_Average_Commission', 'col_13', 'col_14']
✅ Extracted 13 columns for: test2.xlsx::Sheet1


In [0]:
spark_dfs = {}

for sheet_key, col_data in extracted_data_map.items():
    print(f"\n🧱 Converting to Spark DataFrame: {sheet_key}")
    
    try:
        # Enhanced: Skip truly empty column data (no keys or all empty lists)
        if not col_data or all(not v for v in col_data.values()):
            print(f"⚠️ All columns are empty for {sheet_key}, skipping.")
            continue
        
        # Align all columns to same length
        max_len = max(len(v) for v in col_data.values())
        for col in col_data:
            col_data[col] += [None] * (max_len - len(col_data[col]))  # More neutral than ""

        df_pd = pd.DataFrame(col_data)
        df_pd["file_name"] = sheet_key.split("::")[0]
        df_pd["sheet_name"] = sheet_key.split("::")[1]

        df_spark = spark.createDataFrame(df_pd)
        spark_dfs[sheet_key] = df_spark

        print(f"✅ Success: {sheet_key} → {df_spark.count()} rows")

    except Exception as e:
        print(f"❌ Failed to convert {sheet_key} — {e}")




🧱 Converting to Spark DataFrame: test2.xlsx::Sheet1
✅ Success: test2.xlsx::Sheet1 → 6 rows


In [0]:
from pyspark.sql.functions import col, isnan

def flag_data_quality_issues(df, sheet_key: str, threshold_null_pct: float = 0.8):
    """
    Flags columns with potential data quality issues.
    """
    issues = []
    row_count = df.count()
    if row_count == 0:
        return [{"sheet": sheet_key, "issue": "❌ Empty DataFrame"}]

    for field in df.schema.fields:
        name = field.name
        try:
            nulls = df.filter(col(name).isNull() | isnan(col(name))).count()
            null_pct = nulls / row_count

            if null_pct >= threshold_null_pct:
                issues.append({
                    "sheet": sheet_key,
                    "column": name,
                    "issue": f"⚠️ {int(null_pct * 100)}% nulls"
                })

            # Type check for numeric-like data in strings
            if field.dataType.simpleString() == "string":
                numeric_like = df.select(col(name)).rdd.map(lambda x: str(x[0]).replace(',', '').replace('.', '').isdigit()).filter(lambda x: x).count()
                if numeric_like > 0 and numeric_like / row_count > 0.5:
                    issues.append({
                        "sheet": sheet_key,
                        "column": name,
                        "issue": "⚠️ Likely numeric, stored as string"
                    })

        except Exception as e:
            issues.append({
                "sheet": sheet_key,
                "column": name,
                "issue": f"❌ Error checking column — {str(e)}"
            })
    return issues


In [0]:
quality_issues = []
for sheet_key, df in spark_dfs.items():
    quality_issues.extend(flag_data_quality_issues(df, sheet_key))

if quality_issues:
    display(pd.DataFrame(quality_issues))
else:
    print("✅ No data quality issues found.")


sheet,column,issue
test2.xlsx::Sheet1,Business_Line,⚠️ 100% nulls
test2.xlsx::Sheet1,2019,⚠️ 100% nulls
test2.xlsx::Sheet1,2021,"⚠️ Likely numeric, stored as string"
test2.xlsx::Sheet1,2022,"⚠️ Likely numeric, stored as string"
test2.xlsx::Sheet1,2023_Estimate,"⚠️ Likely numeric, stored as string"
test2.xlsx::Sheet1,2023_Revised,"⚠️ Likely numeric, stored as string"
test2.xlsx::Sheet1,2024_Calendar_Year_Estimate,"⚠️ Likely numeric, stored as string"
test2.xlsx::Sheet1,2024_Treaty_Year_Estimate,"⚠️ Likely numeric, stored as string"
test2.xlsx::Sheet1,2024_Q1_Estimate,"⚠️ Likely numeric, stored as string"
test2.xlsx::Sheet1,2025_Q1_Estimate,"⚠️ Likely numeric, stored as string"


In [0]:
# Pick the sheet you want to read (use the exact sheet key)
sheet_key = "test2.xlsx::Sheet1"  # Example

df = spark_dfs[sheet_key]
display(df)  # Show all rows


Business_Line,2019,2020,2021,2022,2023_Estimate,2023_Revised,2024_Calendar_Year_Estimate,2024_Treaty_Year_Estimate,2024_Q1_Estimate,2025_Q1_Estimate,2025_Q2_Estimate,2024_Average_Commission,file_name,sheet_name
nan,nan,Americas R&W,110.2,175.4,310.5,180.3,210,120.6,105.5,140.2,22.1,30.2,test2.xlsx,Sheet1
nan,nan,EMEA W&I,70,82.1,130.2,89.0,95.5,60.4,91.1,120.3,14.2,22.5,test2.xlsx,Sheet1
nan,nan,APAC W&I,35.3,47.5,69.0,42.2,50.2,32.5,40.2,52.3,10.1,12,test2.xlsx,Sheet1
nan,nan,Tax,28.4,22.0,55.6,33.4,38.1,29,46.3,62.1,13.6,21,test2.xlsx,Sheet1
nan,nan,CLRI,14.1,10.5,35.2,20.2,30.3,25,52.4,72.5,9.8,14.4,test2.xlsx,Sheet1
nan,nan,Total,258,337.5,600.5,365.1,424.1,267.5,335.5,447.4,69.8,100.1,test2.xlsx,Sheet1


In [0]:
# # Define where to write (adjust for your catalog/schema)
# base_path = "/Volumes/bdx/data_dictionary/bdx_files"

# for sheet_key, df_spark in spark_dfs.items():
#     try:
#         file_name, sheet_name = sheet_key.split("::")
#         safe_sheet = sheet_name.replace(" ", "_").lower()
#         output_path = f"{base_path}/{file_name}/{safe_sheet}"

#         print(f"📁 Writing Delta table to: {output_path}")
        
#         df_spark.write.format("delta").mode("overwrite").save(output_path)
#         print(f"✅ Delta write complete for: {sheet_key}")
    
#     except Exception as e:
#         print(f"❌ Failed to save Delta table for {sheet_key} — {e}")


📁 Writing Delta table to: /Volumes/bdx/data_dictionary/bdx_files/test2.xlsx/sheet1
✅ Delta write complete for: test2.xlsx::Sheet1


In [0]:
# for sheet_key, df in spark_dfs.items():
#     table_name = "extracted_" + sheet_key.replace(".xlsx", "").replace("::", "_").replace(" ", "_").lower()
#     df.write.format("delta").mode("overwrite").saveAsTable(table_name)
#     print(f"💾 Saved to Delta: {table_name}")